# Retrieve-Then-Reason Evaluation

This notebook runs the upgraded retrieve-then-reason pipeline and inspects:

- overall top-1 and top-3 performance
- per-label accuracy
- row-normalized confusion patterns
- confidence-vs-correctness behavior


In [ ]:
import pandas as pd

from src.ebm_utils import load_knowledge_snippets
from src.evaluation import (
    confidence_accuracy_table,
    confusion_matrix_table,
    normalize_confusion_rows,
    per_label_accuracy,
    plot_confidence_vs_accuracy,
    plot_confusion_matrix,
    summarize_metrics,
)
from src.llm_runner import run_openai
from src.run_experiments import load_split_dataset, run_retrieve_then_reason_case
from src.config import EBM_DIR, OUTPUT_DIR


In [ ]:
full_df, train_df, test_df, label_space = load_split_dataset()
knowledge_snippets_df = load_knowledge_snippets(EBM_DIR / "knowledge_snippets.csv")

label_space[:10], len(test_df)


In [ ]:
records = []

for idx, row in test_df.head(50).iterrows():
    out = run_retrieve_then_reason_case(
        row=row,
        label_space=label_space,
        knowledge_snippets_df=knowledge_snippets_df,
        train_df=train_df,
        llm_call_fn=run_openai,
    )

    records.append({
        "case_id": idx,
        "symptoms": row["symptoms"],
        "true_label": row["diagnosis"],
        "pred_top1": out["pred_top1"],
        "pred_top3": out["pred_top3"],
        "top1_confidence": out["pred_confidences"][0] if out["pred_confidences"] else None,
        "raw_output": out["raw_output"],
        "candidate_labels": out["candidate_labels"],
        "evidence_snippets": out["evidence_snippets"],
        "consultation_guidance": out["parsed"]["consultation_guidance"],
        "ranked_diagnoses": out["parsed"]["ranked_diagnoses"],
    })

results_df = pd.DataFrame(records)
results_df.to_csv(OUTPUT_DIR / "retrieve_then_reason_results.csv", index=False)
results_df.head()


In [ ]:
summary = summarize_metrics(results_df)
per_label = per_label_accuracy(results_df)
labels = sorted(results_df["true_label"].unique().tolist())
cm = confusion_matrix_table(results_df, labels=labels)
cm_norm = normalize_confusion_rows(cm)
conf_table = confidence_accuracy_table(results_df, confidence_col="top1_confidence")

print(summary)
display(per_label)
display(conf_table)
plot_confusion_matrix(cm_norm, title="Retrieve-Then-Reason Confusion Matrix")
plot_confidence_vs_accuracy(conf_table)
